# 3. Advanced Rules

A modernization review: the source is last night's warehouse extract; the target is the same 1,000 NYC taxi trips after a pipeline rewrite. `id` (0–999) is unique and is the primary key. Every mutation is a function of `id`, so the recorded verdicts are stable.

`strict_types=True`. Without it the engine's soft-cast would silently reconcile text codes and padded zone IDs, which hides exactly the type drift a modernization review needs to see. With it, each type-aligning rule is demonstrably necessary.

Load via `veridelta.datasets.load_nyc_taxi()` (cached under `~/.cache/veridelta`).

## 1. The modernized pipeline

Declared variance: sub-cent rounding on fares, a recalculated gratuity, lowercased and padded flags, trip distance exported with a unit suffix, dropoff timestamps as `dd/mm/YYYY`, payment type as text, zero-padded zone codes, legacy rate-code `99` as NULL, `Airport_fee` renamed, two `_etl_*` bookkeeping columns, four trips dropped, three cloned. The defect is a +1 hour pickup shift on every `id % 100 == 7` — ten trips, no rule forgives it.

In [ ]:
import polars as pl

from veridelta import DiffConfig, DiffEngine, DiffRule
from veridelta.datasets import load_nyc_taxi
from veridelta.models import DiffResult

source = load_nyc_taxi()
target = (
    source.with_columns(
        fare_amount=pl.col("fare_amount") + 0.004,
        total_amount=pl.col("total_amount") + 0.004,
        tip_amount=pl.col("tip_amount") * 1.005,
        store_and_fwd_flag=" " + pl.col("store_and_fwd_flag").str.to_lowercase() + " ",
        trip_distance=pl.col("trip_distance").cast(pl.String) + pl.lit(" mi"),
        tpep_dropoff_datetime=pl.col("tpep_dropoff_datetime").dt.strftime("%d/%m/%Y %H:%M:%S"),
        payment_type=pl.col("payment_type").cast(pl.String),
        PULocationID=pl.col("PULocationID").cast(pl.String).str.pad_start(3, "0"),
        RatecodeID=pl.when(pl.col("RatecodeID") == 99).then(None).otherwise(pl.col("RatecodeID")),
        tpep_pickup_datetime=pl.when((pl.col("id") % 100) == 7)
        .then(pl.col("tpep_pickup_datetime") + pl.duration(hours=1))
        .otherwise(pl.col("tpep_pickup_datetime")),
        _etl_batch=pl.lit(1),
        _etl_loaded_at=pl.lit("nightly"),
    )
    .rename({"Airport_fee": "airport_fee"})
    .filter((pl.col("id") % 250) != 0)
)
clones = target.head(3).with_columns(id=pl.Series([1000, 1001, 1002], dtype=pl.Int64))
target = pl.concat([target, clones], how="vertical")
source_lf, target_lf = source.lazy(), target.lazy()
print(f"source={source.height}  target={target.height}")


def verdict(label: str, result: DiffResult) -> None:
    """Print one run's status, row counts, and columns ranked by drift."""
    summary = result.summary
    status = "PASSED" if summary.is_match else "FAILED"
    ranked = (
        ", ".join(
            f"{name}={count}"
            for name, count in sorted(
                summary.column_mismatches.items(), key=lambda item: (-item[1], item[0])
            )
        )
        or "none"
    )
    print(
        f"{label}: {status}  changed={summary.changed_count}  "
        f"added={summary.added_count}  removed={summary.removed_count}  drift=[{ranked}]"
    )


# Output:
# source=1000  target=999

## 2. Baseline

Default `schema_mode` is `intersection`. `Airport_fee` / `airport_fee` are not on both sides, so the rename is silently skipped. Nearly every overlapping row is changed.

In [ ]:
baseline = DiffEngine(
    DiffConfig(primary_keys=["id"], strict_types=True), source_lf, target_lf
).run()
verdict("1 baseline", baseline)
print("airport_fee compared:", "airport_fee" in baseline.compared_columns)
print("Airport_fee compared:", "Airport_fee" in baseline.compared_columns)

# Output:
# 1 baseline: FAILED  changed=996  added=3  removed=4  drift=[PULocationID=996, fare_amount=996, payment_type=996, store_and_fwd_flag=996, total_amount=996, tpep_dropoff_datetime=996, trip_distance=996, tip_amount=770, tpep_pickup_datetime=10, RatecodeID=3]
# airport_fee compared: False
# Airport_fee compared: False

## 3. Value rules

A `pattern=r".*_amount$"` absolute tolerance of one cent covers fare and total. An exact-name `tip_amount` relative tolerance of 1% wins by precedence. Whitespace and case on the flag, regex plus cast on trip distance, and `null_values=[99]` on the rate code. Drift shrinks to the type and time columns.

In [ ]:
value_rules = [
    DiffRule(pattern=r".*_amount$", absolute_tolerance=0.01),
    DiffRule(column_names=["tip_amount"], relative_tolerance=0.01),
    DiffRule(
        column_names=["store_and_fwd_flag"],
        whitespace_mode="both",
        case_insensitive=True,
    ),
    DiffRule(
        column_names=["trip_distance"],
        regex_replace={r"\s*mi$": ""},
        cast_to="Float64",
    ),
    DiffRule(column_names=["RatecodeID"], null_values=[99]),
]
values = DiffEngine(
    DiffConfig(primary_keys=["id"], strict_types=True, rules=value_rules),
    source_lf,
    target_lf,
).run()
verdict("2 value rules", values)

# Output:
# 2 value rules: FAILED  changed=996  added=3  removed=4  drift=[PULocationID=996, payment_type=996, tpep_dropoff_datetime=996, tpep_pickup_datetime=10]

## 4. Type and time rules

`cast_to`, `pad_zeros`, and `datetime_format` align the remaining declared type drift. Residual value drift is the pickup-hour shift.

In [ ]:
type_rules = [
    *value_rules,
    DiffRule(column_names=["payment_type"], cast_to="Int64"),
    DiffRule(column_names=["PULocationID"], pad_zeros=3),
    DiffRule(column_names=["tpep_dropoff_datetime"], datetime_format="%d/%m/%Y %H:%M:%S"),
]
typed = DiffEngine(
    DiffConfig(primary_keys=["id"], strict_types=True, rules=type_rules),
    source_lf,
    target_lf,
).run()
verdict("3 type and time", typed)

# Output:
# 3 type and time: FAILED  changed=10  added=3  removed=4  drift=[tpep_pickup_datetime=10]

## 5. Structure

`rename_to` maps `Airport_fee` onto `airport_fee`. `pattern="^_etl_"` ignore plus `schema_mode="allow_additions"` absorbs the bookkeeping columns. The residual is the finding.

In [ ]:
structure_rules = [
    *type_rules,
    DiffRule(column_names=["Airport_fee"], rename_to="airport_fee"),
    DiffRule(pattern=r"^_etl_", ignore=True),
]
final = DiffEngine(
    DiffConfig(
        primary_keys=["id"],
        strict_types=True,
        schema_mode="allow_additions",
        rules=structure_rules,
    ),
    source_lf,
    target_lf,
).run()
verdict("4 structure", final)
print(final.get_mismatches("tpep_pickup_datetime"))

# Output:
# 4 structure: FAILED  changed=10  added=3  removed=4  drift=[tpep_pickup_datetime=10]
# shape: (10, 3)
# ┌─────┬─────────────────────────────┬─────────────────────────────┐
# │ id  ┆ tpep_pickup_datetime_source ┆ tpep_pickup_datetime_target │
# │ --- ┆ ---                         ┆ ---                         │
# │ i64 ┆ datetime[μs]                ┆ datetime[μs]                │
# ╞═════╪═════════════════════════════╪═════════════════════════════╡
# │ 7   ┆ 2026-01-01 00:34:28         ┆ 2026-01-01 01:34:28         │
# │ 107 ┆ 2026-01-01 00:20:33         ┆ 2026-01-01 01:20:33         │
# │ 207 ┆ 2026-01-01 00:58:44         ┆ 2026-01-01 01:58:44         │
# │ 307 ┆ 2026-01-01 00:24:47         ┆ 2026-01-01 01:24:47         │
# │ 407 ┆ 2026-01-01 00:58:52         ┆ 2026-01-01 01:58:52         │
# │ 507 ┆ 2026-01-01 00:28:51         ┆ 2026-01-01 01:28:51         │
# │ 607 ┆ 2026-01-01 00:22:13         ┆ 2026-01-01 01:22:13         │
# │ 707 ┆ 2026-01-01 00:14:57         ┆ 2026-01-01 01:14:57         │
# │ 807 ┆ 2026-01-01 00:05:21         ┆ 2026-01-01 01:05:21         │
# │ 907 ┆ 2026-01-01 00:59:11         ┆ 2026-01-01 01:59:11         │
# └─────┴─────────────────────────────┴─────────────────────────────┘

Ten trips, each shifted by exactly one hour. A `threshold` of `0.02` would turn this green: 10 changed + 4 removed + 3 added is a 1.7% mismatch ratio on 1,000 source rows. That is the wrong call. Exactness means the rules absorb declared variance and what remains is a finding.

## 6. YAML for CI

The equivalent policy for [2. YAML and CLI](02_yaml_and_cli.ipynb). The HTML hand-off is [4. HTML Reports](04_html_reports.ipynb).

In [ ]:
import yaml

print(
    yaml.safe_dump(
        DiffConfig(
            primary_keys=["id"],
            strict_types=True,
            schema_mode="allow_additions",
            rules=structure_rules,
        ).model_dump(mode="json", exclude_none=True, exclude_defaults=True),
        sort_keys=False,
    )
)

# Output:
# primary_keys:
# - id
# schema_mode: allow_additions
# strict_types: true
# rules:
# - pattern: .*_amount$
#   absolute_tolerance: 0.01
# - column_names:
#   - tip_amount
#   relative_tolerance: 0.01
# - column_names:
#   - store_and_fwd_flag
#   case_insensitive: true
#   whitespace_mode: both
# - column_names:
#   - trip_distance
#   regex_replace:
#     \s*mi$: ''
#   cast_to: Float64
# - column_names:
#   - RatecodeID
#   null_values:
#   - 99
# - column_names:
#   - payment_type
#   cast_to: Int64
# - column_names:
#   - PULocationID
#   pad_zeros: 3
# - column_names:
#   - tpep_dropoff_datetime
#   datetime_format: '%d/%m/%Y %H:%M:%S'
# - column_names:
#   - Airport_fee
#   rename_to: airport_fee
# - pattern: ^_etl_
#   ignore: true